# OpenAI Swarm Demo

This notebook builds a small agent swarm using `AgentBuilder`, shared capabilities,
and the OpenAI adapter. Set your API key below, then execute the swarm cell to see
a researcher and strategist collaborate like lego pieces.

In [ ]:
import os
from getpass import getpass

from agentic_codex import AgentBuilder, Context, EnvOpenAIAdapter, SwarmCoordinator
from agentic_codex.core.memory import EpisodicMemory
from agentic_codex.core.schemas import AgentStep, Message

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ")  # noqa: S603

In [ ]:
KNOWLEDGE_BASE = [
    {
        "topic": "andromeda galaxy",
        "answer": "The Andromeda Galaxy (M31) is the nearest major galaxy to the Milky Way and on a long-term collision course with it.",
    },
    {
        "topic": "observation",
        "answer": "Under dark skies, Andromeda is visible to the naked eye as a faint smudge; binoculars reveal its disc and companion galaxies.",
    },
    {
        "topic": "astrophotography",
        "answer": "Amateur astronomers value Andromeda because its large apparent size makes it an accessible target for wide-field astrophotography.",
    },
]

shared_adapter = EnvOpenAIAdapter(model="gpt-4o-mini")


def build_research_agent(adapter: EnvOpenAIAdapter) -> AgentBuilder:
    def researcher_step(ctx: Context) -> AgentStep:
        knowledge = ctx.components["context"]["knowledge"]
        persona = ctx.components["persona.researcher"]
        context_lines = "\n".join(f"- {item['topic'].title()}: {item['answer']}" for item in knowledge)
        prompt = (
            f"{persona}\n"
            f"Goal: {ctx.goal}\n"
            "Context:\n"
            f"{context_lines}\n"
            "Return three concise research findings that highlight why the topic matters."
        )
        notes = ctx.llm.generate(prompt)
        ctx.scratch["research_notes"] = notes
        memory = ctx.components["memory:research"]
        memory.put("notes", notes)
        return AgentStep(
            out_messages=[Message(role="assistant", content=notes, meta={"agent": "researcher"})]
        )

    return (
        AgentBuilder(name="researcher", role="assistant")
        .with_llm(adapter)
        .with_context({"knowledge": KNOWLEDGE_BASE})
        .with_memory(EpisodicMemory(), slot="research")
        .with_resource(
            "persona.researcher",
            "You are a curious astrophysicist collecting evidence about nearby galaxies.",
        )
        .with_step(researcher_step)
    )


def build_strategy_agent(adapter: EnvOpenAIAdapter) -> AgentBuilder:
    def strategist_step(ctx: Context) -> AgentStep:
        persona = ctx.components["persona.strategist"]
        research_notes = ctx.scratch.get("research_notes", "")
        prompt = (
            f"{persona}\n"
            f"Goal: {ctx.goal}\n"
            "Research notes:\n"
            f"{research_notes}\n"
            "Produce an outreach action plan with three steps and a short risk/mitigation note."
        )
        plan = ctx.llm.generate(prompt)
        ctx.scratch["strategy_plan"] = plan
        memory = ctx.components["memory:strategy"]
        memory.put("plan", plan)
        return AgentStep(
            out_messages=[Message(role="assistant", content=plan, meta={"agent": "strategist"})]
        )

    return (
        AgentBuilder(name="strategist", role="assistant")
        .with_llm(adapter)
        .with_context({"knowledge": KNOWLEDGE_BASE})
        .with_memory(EpisodicMemory(), slot="strategy")
        .with_resource(
            "persona.strategist",
            "You are an outreach strategist translating research into compelling public experiences.",
        )
        .with_step(strategist_step)
    )


def make_openai_aggregator(adapter: EnvOpenAIAdapter, goal: str):
    def aggregator(messages):
        digest = "\n\n".join(
            f"{msg.meta.get('agent', msg.role).title()} says:\n{msg.content}" for msg in messages
        )
        prompt = (
            "You are the swarm coordinator combining agent insights into a final response.\n"
            f"Goal: {goal}\n"
            "Agent contributions:\n"
            f"{digest}\n"
            "Write a structured summary with a headline, three key bullets, and a concrete next step."
        )
        summary = adapter.generate(prompt)
        return Message(
            role="aggregator",
            content=summary,
            meta={"sources": [msg.meta.get("agent", msg.role) for msg in messages]},
        )

    return aggregator

In [ ]:
GOAL = "Design a short public talk explaining why the Andromeda Galaxy matters to amateur astronomers."

researcher_agent = build_research_agent(shared_adapter).build()
strategist_agent = build_strategy_agent(shared_adapter).build()

swarm = SwarmCoordinator(
    [researcher_agent, strategist_agent],
    aggregator=make_openai_aggregator(shared_adapter, GOAL),
)

result = swarm.run(goal=GOAL, inputs={})

for message in result.messages[:-1]:
    print(f"{message.meta.get('agent', message.role).title()} Output:\n{message.content}\n---\n")

final_summary = result.messages[-1].content
final_summary